# SMILES VAE demo

Train a small convolutional VAE on a few SMILES strings and generate new ones using the `smiles_vae` package.

[Open in Colab](https://colab.research.google.com/drive/1VCyw-wDqRv2e03lCjmeXkNhdD7By6lGW?usp=sharing) (original notebook).

## 1. Setup
Run from the repository root, or install the package first with `pip install -e .`.

In [ ]:
# Uncomment on Colab:
# !pip install -q rdkit tensorflow matplotlib
# !pip install -q git+https://github.com/bhargavmanchala/Drug_Discovery.git
import matplotlib.pyplot as plt

from smiles_vae.analysis import describe, validity_rate
from smiles_vae.data import DEFAULT_SMILES
from smiles_vae.pipeline import generate, train

## 2. Train
SMILES are one-hot encoded with an explicit pad token, then fed to the VAE.

In [ ]:
t = train(DEFAULT_SMILES, epochs=25, batch_size=2, seed=42)
t.history['loss'][-1]

## 3. Loss curves

In [ ]:
plt.plot(t.history['loss'], label='Training')
plt.plot(t.history['val_loss'], label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

## 4. Generate and analyse
Sample from the latent prior, decode, and check validity with RDKit.

In [ ]:
new = generate(t, n=10, seed=42)
print(new)
print(f'Validity: {validity_rate(new):.0%}')
for s in new:
    d = describe(s)
    print(s, '->', 'invalid' if d is None else f'MW={d.mol_wt:.2f} LogP={d.log_p:.2f} HBD={d.h_donors} HBA={d.h_acceptors}')

## Notes
With only five training molecules the model memorises or collapses, and many outputs are chemically invalid. Use a larger dataset (for example ZINC or ChEMBL subsets) for meaningful results.